# Data Preprocessing Strategy
**Goal:** Extract domain invariant features, construct robust ground truth (Health Index & RUL labels), apply sequence sliding windows, and prepare the final tensor dataframes for Deep Learning evaluation flow.
*   **Step 1:** Feature Extraction (Maximum FFT bandwidth capped at 1280 Hz to align dimensions with source domain).
*   **Step 2:** Construct Ground Truth Targets. Employs exact CUSUM logic (`calculate_series_health_index`) where $HI$ operates on a pristine $[1.0 \rightarrow 0.0]$ boundary.
*   **Step 3:** Sliding Window Mapping & Tensor Compilation. Generate metadata for lifespan reference.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import random
from typing import List, Dict, Tuple
from tqdm import tqdm

# Add local source to path for imports
sys.path.append("../src")
from CrossDomainFeatureExtractor import CrossDomainFeatureExtractor
from ConstructGroundTruthUsingCUSUM import UnivariateCUSUMDetector

# ==========================================
# CONFIGURATION
# ==========================================
# Base directories
INPUT_PATH = r"D:\Proyek Dosen\Riset Bearing\XJTU-SY_Bearing_Datasets\Processed_Data\Downsampled"
OUTPUT_PATH = r"D:\Proyek Dosen\Riset Bearing\XJTU-SY_Bearing_Datasets\Processed_Data\LSTM_Inputs"

os.makedirs(OUTPUT_PATH, exist_ok=True)

# Dynamic Window Sizes for Tensor Mapping
WINDOW_SIZES = [10, 20, 30] 

# Domain Config
XJTU_SAMPLING_RATE = 25600.0
MAX_FREQ_HZ = 1280.0 # Strict Nyquist Alignment

# Conditions map
CONDITIONS = ['35Hz12kN', '37.5Hz11kN', '40Hz10kN']

print(f"Input Path: {os.path.abspath(INPUT_PATH)}")
print(f"Output Path: {os.path.abspath(OUTPUT_PATH)}")
print(f"Window Sizes: {WINDOW_SIZES}")

In [ ]:
def calculate_series_health_index(current_time: float, turning_point_time: float, total_lifespan: float) -> float:
    """
    Calculate the health index based on current time, turning point (Tcp), and total lifespan (Tf).
    Returns a health index value strictly bounded between 1.0 (Healthy) and 0.0 (Failed).
    Maintains constant maximum health before the degradation point.
    """
    if total_lifespan == turning_point_time:
        return 0.0  # Prevent division by zero
    if current_time <= turning_point_time:
        return 1.0
    return 1.0 - ((current_time - turning_point_time) / (total_lifespan - turning_point_time))

def create_sliding_windows(data_df: pd.DataFrame, window_size: int) -> pd.DataFrame:
    """
    Transforms continuous 1D features into flattened 2D sequence arrays.
    Returns: DataFrame containing windowed rows that can later be parsed with 
    `.reshape(N, F, W).transpose(0, 2, 1)` to prevent tensor scrambling.
    """
    if len(data_df) <= window_size:
        return pd.DataFrame()

    windows = []
    features_col = [c for c in data_df.columns if c not in ["Change_Point", "Health_Index", "Target_RUL", "Minute", "Bearing_ID"]]
    
    for i in range(len(data_df) - window_size):
        window_segment = data_df.iloc[i : i + window_size]
        
        row_record = {}
        # Flatten by feature first so it reconstructs efficiently with reshape(N, F, W)
        for feat in features_col:
            for t in range(window_size):
                row_record[f"{feat}_t{t}"] = window_segment.iloc[t][feat]
                
        # Targets are taken from the LAST timestamp in the window
        row_record['Health_Index'] = window_segment.iloc[-1]['Health_Index']
        row_record['Target_RUL'] = window_segment.iloc[-1]['Target_RUL']
        row_record['Bearing_ID'] = window_segment.iloc[-1]['Bearing_ID']
        row_record['Original_Minute'] = window_segment.iloc[-1]['Minute']
        windows.append(row_record)
        
    return pd.DataFrame(windows)

def extract_features_and_build_gt(file_path: str, bearing_id: str) -> Tuple[pd.DataFrame, Dict]:
    """
    1. Extracts Features per minute. 
    2. Aligns Negative Transfer Trends.
    3. Normalizes using Healthy Baseline (First 10%).
    4. Constructs Ground Truth utilizing CUSUM point estimation.
    """
    df_raw = pd.read_pickle(file_path)
    
    # 1. Feature Extraction
    extractor = CrossDomainFeatureExtractor(sampling_rate=XJTU_SAMPLING_RATE, max_freq_hz=MAX_FREQ_HZ)
    features_list = []
    
    for idx, row in tqdm(df_raw.iterrows(), total=len(df_raw), desc=f"Ext. {bearing_id}", leave=False):
        try:
            h_sig = np.array(row['H_Signal'], dtype=float)
        except Exception:
            h_sig = np.zeros(32768)
            
        feats = extractor.extract_all_features(h_sig)
        feats['Minute'] = row['Minute']
        feats['Bearing_ID'] = bearing_id
        features_list.append(feats)
        
    df_features = pd.DataFrame(features_list)
    feat_cols = [c for c in df_features.columns if c not in ['Minute', 'Bearing_ID']]
    
    # 2. CUSUM Ground Truth Estimation on raw RMS to detect defect inflection
    detector = UnivariateCUSUMDetector(baseline_ratio=0.1, k_factor=0.5, h_factor=5.0)
    # Get rms feature correctly mapped
    rms_col = "td_rms" if "td_rms" in df_features.columns else feat_cols[0]
    change_point, _ = detector.fit_predict(df_features[rms_col].values)
    
    total_life = len(df_features)
    health_indices = []
    rul_labels = []
    
    for min_idx in range(total_life):
        minute = min_idx + 1
        hi = calculate_series_health_index(minute, change_point, total_life)
        health_indices.append(hi)
        max_rul = total_life - change_point
        rul_labels.append(min(total_life - minute, max_rul))
            
    df_features['Change_Point'] = change_point
    df_features['Health_Index'] = health_indices
    df_features['Target_RUL'] = rul_labels

    # 3. Cross-Domain Feature Trend Alignment (Negative Transfer Mitigation)
    for col in feat_cols:
        correlation = df_features[col].corr(df_features['Health_Index'])
        # If correlation is positive, feature decreases as HI decreases (bad for target matching domain alignment).
        if correlation > 0:
            df_features[col] = df_features[col] * -1.0

    # 4. Healthy Baseline Normalization (NO MinMaxScaler or RobustScaler)
    healthy_len = max(1, int(0.1 * len(df_features))) # Guaranteed healthy first 10%
    healthy_baseline = df_features.iloc[:healthy_len]
    h_mean = healthy_baseline[feat_cols].mean()
    h_std = healthy_baseline[feat_cols].std()
    
    # Deal with zeroes in standard deviation dynamically
    h_std = h_std.replace(0.0, 1e-8)
    
    df_features[feat_cols] = (df_features[feat_cols] - h_mean) / h_std

    metadata = {
        "Bearing_ID": bearing_id,
        "Total_Lifespan": total_life,
        "Change_Point": change_point
    }
    
    return df_features, metadata

def get_stratified_folds() -> List[Dict[str, List[str]]]:
    """Generates the absolute 5 Folds OC-Independent validation mapping."""
    # Assuming bearing naming structure Condition_Idx
    folds = [
        {"val": ["Bearing1_1", "Bearing2_1", "Bearing3_1"]},
        {"val": ["Bearing1_2", "Bearing2_2", "Bearing3_2"]},
        {"val": ["Bearing1_3", "Bearing2_3", "Bearing3_3"]},
        {"val": ["Bearing1_4", "Bearing2_4", "Bearing3_4"]},
        {"val": ["Bearing1_5", "Bearing2_5", "Bearing3_5"]}
    ]
    all_bearings = [f"Bearing{c}_{i}" for c in range(1, 4) for i in range(1, 6)]
    
    for fold in folds:
        fold["train"] = [b for b in all_bearings if b not in fold["val"]]
        
    return folds

def normalize_bearing_name(raw_name: str) -> str:
    """Standardizes input labels (e.g. '35Hz12kN_Bearing1_1' -> 'Bearing1_1')."""
    if "Bearing" in raw_name:
        parts = raw_name.split("Bearing")
        if len(parts) > 1:
            return "Bearing" + parts[1]
    return raw_name

def preprocess_and_slide():
    if not os.path.exists(INPUT_PATH):
        print(f"Directory {INPUT_PATH} not found.")
        return
        
    all_bearings_features = {}
    all_metadata = []
    
    import glob
    downsampled_files = glob.glob(os.path.join(INPUT_PATH, "*", "*_downsampled.pkl"))
    
    if not downsampled_files:
        print("No Pickle Files found. Process aborted. Make sure to run Notebook 1 first!")
        return
    
    # Step 1 & 2: Feature Extraction and Ground Truth
    print("--- Feature Extraction & Baseline Normalization Phase ---")
    for file in tqdm(downsampled_files, desc="Processing Bearings"):
        raw_b_id = os.path.basename(file).replace("_downsampled.pkl", "")
        # Try finding standard bearing ID matching Fold definitions
        bearing_id = normalize_bearing_name(raw_b_id)
        
        # Override for dataset specifics: map XJTU to 'BearingC_I' format
        # Conditions map directly to 1, 2, 3
        # Folder is file's parent
        parent_dir = os.path.basename(os.path.dirname(file))
        c_idx = 1
        if '37.5' in parent_dir: c_idx = 2
        elif '40' in parent_dir: c_idx = 3
        
        # Usually XJTU bearings are just named Bearing1_1 etc. Let's guarantee alignment:
        if raw_b_id.startswith("Bearing"):
            b_num = raw_b_id.split("_")[-1]
            bearing_id = f"Bearing{c_idx}_{b_num}"
        else:
            bearing_id = raw_b_id

        df_feat, metadata = extract_features_and_build_gt(file, bearing_id)
        df_feat['Normalized_Bearing_ID'] = bearing_id # Ensuring unified mapping
        all_bearings_features[bearing_id] = df_feat
        all_metadata.append(metadata)

    # Export Metadata Reference File required during Training/Val evaluation
    metadata_df = pd.DataFrame(all_metadata)
    metadata_path = os.path.join(OUTPUT_PATH, "bearing_metadata.csv")
    metadata_df.to_csv(metadata_path, index=False)
    
    folds = get_stratified_folds()

    # Step 3: Dynamic Windows & Stratified CV Export
    print("\n--- Sliding Windows & 5-Fold Stratified OC-Independent Split ---")
    for w_size in WINDOW_SIZES:
        W_DIR = os.path.join(OUTPUT_PATH, f"window_size_{w_size}")
        os.makedirs(W_DIR, exist_ok=True)
        
        # Fast Window Generation over dataset
        windowed_dfs = {}
        for b_id, df_feat in all_bearings_features.items():
            w_df = create_sliding_windows(df_feat, w_size)
            if not w_df.empty:
                windowed_dfs[b_id] = w_df
            else:
                print(f"[Warning] Skipped {b_id} (Insuff length {len(df_feat)})")

        # Compile and generate exact Folds sequentially
        for fold_idx, cv_split in enumerate(folds, start=1):
            val_bearings = cv_split['val']
            train_bearings = cv_split['train']
            
            # Form Validation Set (no shuffle to maintain temporal sequencing for plots)
            val_list = [windowed_dfs[b] for b in val_bearings if b in windowed_dfs]
            if val_list:
                df_val = pd.concat(val_list, ignore_index=True)
                val_out = os.path.join(W_DIR, f"processed_val_fold{fold_idx}.csv")
                df_val.to_csv(val_out, index=False)
            
            # Form Training Set (shuffled temporally)
            train_list = [windowed_dfs[b] for b in train_bearings if b in windowed_dfs]
            if train_list:
                df_train = pd.concat(train_list, ignore_index=True)
                df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)
                train_out = os.path.join(W_DIR, f"processed_train_fold{fold_idx}.csv")
                df_train.to_csv(train_out, index=False)
                
            print(f"[W={w_size}] Fold {fold_idx}: Saved {len(train_bearings)} Train and {len(val_bearings)} Val Bearings")

if __name__ == "__main__":
    preprocess_and_slide()